# Issue #4: Raw Data Preprocessing for Traffic Demand Analysis

This notebook prepares raw 15-minute SVC volume records from **2015-2019** into stable daily and hourly traffic flow datasets for traffic demand analysis, peak-period analysis, and downstream signal timing optimization.


## Research Context

The project now focuses on **data-driven traffic signal timing optimization for smart cities** using the City of Toronto's pre-COVID baseline period. Raw 15-minute counts remain the source of truth because they preserve operational demand patterns more directly than summary-only abstractions.


## 1) Setup and Path Resolution

We detect the repository root, resolve the raw input files from the updated `data/raw/traffic/` layout, and define processed outputs for both daily traffic flow analysis and hourly optimization workflows. Signal reference files now live under `data/raw/signals/` for later baseline timing and simulation notebooks.


In [1]:
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    return start


repo_root = find_repo_root(Path.cwd())
print(f"Repo root: {repo_root}")

raw_candidates = [
    repo_root / "data" / "raw" / "traffic" / "svc_raw_data_volume_2015_2019.csv",
    repo_root / "data" / "raw" / "svc_raw_data_volume_2015_2019.csv",
    Path("/mnt/data/svc_raw_data_volume_2015_2019.csv"),
]

summary_candidates = [
    repo_root / "data" / "raw" / "traffic" / "svc_summary_data.csv",
    repo_root / "data" / "raw" / "svc_summary_data.csv",
    Path("/mnt/data/svc_summary_data.csv"),
]

signal_candidates = {
    "csv": [
        repo_root / "data" / "raw" / "signals" / "traffic_signals.csv",
    ],
    "geojson": [
        repo_root / "data" / "raw" / "signals" / "traffic_signals.geojson",
    ],
}

raw_path_opt: Optional[Path] = next((p for p in raw_candidates if p.exists()), None)
if raw_path_opt is None:
    raise FileNotFoundError(f"Could not find raw file in candidates: {raw_candidates}")
raw_path: Path = raw_path_opt

summary_path: Optional[Path] = next((p for p in summary_candidates if p.exists()), None)
signal_paths = {name: next((p for p in paths if p.exists()), None) for name, paths in signal_candidates.items()}

processed_dir = repo_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

daily_output_path = processed_dir / "modeling_dataset_2015_2019.csv"
hourly_output_path = processed_dir / "traffic_signal_hourly_dataset_2015_2019.csv"

print(f"Using raw input: {raw_path}")
print(f"Summary cross-check input: {summary_path}")
print(f"Signal CSV reference: {signal_paths['csv']}")
print(f"Signal GeoJSON reference: {signal_paths['geojson']}")
print(f"Daily output path: {daily_output_path}")
print(f"Hourly output path: {hourly_output_path}")


Repo root: /Users/manavparikh/Documents/GitHub/unfc-capstone-traffic-forecasting
Using raw input: /Users/manavparikh/Documents/GitHub/unfc-capstone-traffic-forecasting/data/raw/traffic/svc_raw_data_volume_2015_2019.csv
Summary cross-check input: /Users/manavparikh/Documents/GitHub/unfc-capstone-traffic-forecasting/data/raw/traffic/svc_summary_data.csv
Signal CSV reference: /Users/manavparikh/Documents/GitHub/unfc-capstone-traffic-forecasting/data/raw/signals/traffic_signals.csv
Signal GeoJSON reference: /Users/manavparikh/Documents/GitHub/unfc-capstone-traffic-forecasting/data/raw/signals/traffic_signals.geojson
Daily output path: /Users/manavparikh/Documents/GitHub/unfc-capstone-traffic-forecasting/data/processed/modeling_dataset_2015_2019.csv
Hourly output path: /Users/manavparikh/Documents/GitHub/unfc-capstone-traffic-forecasting/data/processed/traffic_signal_hourly_dataset_2015_2019.csv


## 2) Load and Clean Raw 15-Minute Data

Cleaning rules:
- parse timestamps from `time_start`
- convert `volume_15min` to numeric
- drop rows with invalid timestamps or traffic counts
- remove duplicate records
- restrict to the 2015-2019 pre-COVID baseline period


In [2]:
assert raw_path is not None
raw_df = pd.read_csv(raw_path)
print(f"Raw shape before cleaning: {raw_df.shape}")
print("Columns:", raw_df.columns.tolist())

if "centreline_id" in raw_df.columns:
    centreline_source = raw_df["centreline_id"]
else:
    centreline_source = pd.Series(np.nan, index=raw_df.index)

centreline = pd.to_numeric(centreline_source, errors="coerce").astype("Int64")
base_loc = centreline.astype(str)
missing_base = centreline.isna()

if "location_name" in raw_df.columns:
    location_name_series = raw_df["location_name"].astype(str).str.strip()
else:
    location_name_series = pd.Series("UNKNOWN", index=raw_df.index, dtype="string")

base_loc = base_loc.mask(missing_base, location_name_series)

if "direction" in raw_df.columns:
    direction = raw_df["direction"].astype(str).str.strip().replace({"": "UNK"})
else:
    direction = pd.Series("UNK", index=raw_df.index, dtype="string")

raw_df["direction"] = direction
raw_df["location_id"] = base_loc + "_" + direction
raw_df["time_start"] = pd.to_datetime(raw_df["time_start"], errors="coerce")
raw_df["volume_15min"] = pd.to_numeric(raw_df["volume_15min"], errors="coerce")

before_drop = len(raw_df)
raw_df = raw_df.dropna(subset=["time_start", "volume_15min"]).copy()
after_drop = len(raw_df)

raw_df = raw_df[raw_df["time_start"].dt.year.between(2015, 2019)].copy()

before_dupes = len(raw_df)
raw_df = raw_df.drop_duplicates(subset=["location_id", "time_start", "volume_15min"])
after_dupes = len(raw_df)

print(f"Dropped invalid datetime/volume rows: {before_drop - after_drop:,}")
print(f"Removed duplicate 15-minute observations: {before_dupes - after_dupes:,}")
print(f"Cleaned raw shape: {raw_df.shape}")
print(f"Year range: {raw_df['time_start'].dt.year.min()}-{raw_df['time_start'].dt.year.max()}")


Raw shape before cleaning: (556608, 10)
Columns: ['id', 'count_id', 'location_name', 'longitude', 'latitude', 'centreline_id', 'time_start', 'time_end', 'direction', 'volume_15min']


Dropped invalid datetime/volume rows: 0
Removed duplicate 15-minute observations: 199
Cleaned raw shape: (556409, 11)
Year range: 2015-2019


## 3) Aggregate 15-Minute Data into Hourly and Daily Traffic Flow Datasets

Aggregation logic:
- **Hourly dataset:** sums all 15-minute records per `location_id` and hour
- **Daily dataset:** summarizes hourly totals by location and date
- **Peak-hour metrics:** identify the highest hourly demand within each location-day
- **Optimization support fields:** preserve hourly share, observed-hour coverage, and demand concentration indicators


In [3]:
raw_df["hour"] = raw_df["time_start"].dt.floor("h")

hourly = (
    raw_df.groupby(["location_id", "hour"], as_index=False)
    .agg(
        hourly_volume=("volume_15min", "sum"),
        location_name=("location_name", "first"),
        centreline_id=("centreline_id", "first"),
        direction=("direction", "first"),
    )
)

hourly["date"] = hourly["hour"].dt.floor("D")
hourly["hour_of_day"] = hourly["hour"].dt.hour
hourly["year"] = hourly["date"].dt.year
hourly["month"] = hourly["date"].dt.month
hourly["day_of_week"] = hourly["date"].dt.dayofweek
hourly["is_weekend"] = hourly["day_of_week"].isin([5, 6]).astype(int)

daily = (
    hourly.groupby(["location_id", "date"], as_index=False)
    .agg(
        location_name=("location_name", "first"),
        centreline_id=("centreline_id", "first"),
        direction=("direction", "first"),
        year=("year", "first"),
        month=("month", "first"),
        day_of_week=("day_of_week", "first"),
        is_weekend=("is_weekend", "first"),
        daily_total_volume=("hourly_volume", "sum"),
        peak_hour_volume=("hourly_volume", "max"),
        observed_hour_count=("hour", "nunique"),
    )
)

daily["avg_observed_hourly_volume"] = np.where(
    daily["observed_hour_count"] > 0,
    daily["daily_total_volume"] / daily["observed_hour_count"],
    np.nan,
)
daily["peak_ratio"] = np.where(
    daily["daily_total_volume"] > 0,
    daily["peak_hour_volume"] / daily["daily_total_volume"],
    np.nan,
)
daily["peak_hour_concentration"] = np.where(
    daily["avg_observed_hourly_volume"] > 0,
    daily["peak_hour_volume"] / daily["avg_observed_hourly_volume"],
    np.nan,
)
daily["queue_pressure_proxy"] = (
    daily["peak_hour_volume"] - daily["avg_observed_hourly_volume"]
).clip(lower=0)

hourly = hourly.merge(
    daily[
        [
            "location_id",
            "date",
            "daily_total_volume",
            "peak_hour_volume",
            "peak_ratio",
            "peak_hour_concentration",
            "observed_hour_count",
        ]
    ],
    on=["location_id", "date"],
    how="left",
)
hourly["hourly_share_of_daily_volume"] = np.where(
    hourly["daily_total_volume"] > 0,
    hourly["hourly_volume"] / hourly["daily_total_volume"],
    np.nan,
)
hourly["is_peak_hour"] = (hourly["hourly_volume"] == hourly["peak_hour_volume"]).astype(int)
hourly["estimated_arrival_rate_vph"] = hourly["hourly_volume"]
hourly["estimated_arrival_rate_vpm"] = hourly["hourly_volume"] / 60.0

daily = daily.sort_values(["location_id", "date"]).reset_index(drop=True)
hourly = hourly.sort_values(["location_id", "hour"]).reset_index(drop=True)

print(f"Hourly shape: {hourly.shape}")
print(f"Daily shape: {daily.shape}")
display(daily.head())
display(hourly.head())


Hourly shape: (138672, 21)
Daily shape: (5778, 16)


,location_id,date,location_name,centreline_id,direction,year,month,day_of_week,is_weekend,daily_total_volume,peak_hour_volume,observed_hour_count,avg_observed_hourly_volume,peak_ratio,peak_hour_concentration,queue_pressure_proxy
0,10010625_WB,2015-05-14,Danforth Ave: Donlands Ave - Byron Ave,10010625,WB,2015,5,3,0,17031,1809,24,709.625000,0.106218,2.549234,1099.375000
1,10010625_WB,2015-05-15,Danforth Ave: Donlands Ave - Byron Ave,10010625,WB,2015,5,4,0,17365,1734,24,723.541667,0.099856,2.396545,1010.458333
2,10010625_WB,2015-05-16,Danforth Ave: Donlands Ave - Byron Ave,10010625,WB,2015,5,5,1,14470,1066,24,602.916667,0.073670,1.768072,463.083333
3,10010625_WB,2015-05-17,Danforth Ave: Donlands Ave - Byron Ave,10010625,WB,2015,5,6,1,12235,973,24,509.791667,0.079526,1.908623,463.208333
4,10010625_WB,2015-05-18,Danforth Ave: Donlands Ave - Byron Ave,10010625,WB,2015,5,0,0,16523,1775,24,688.458333,0.107426,2.578224,1086.541667


,location_id,hour,hourly_volume,location_name,centreline_id,direction,date,hour_of_day,year,month,day_of_week,is_weekend,daily_total_volume,peak_hour_volume,peak_ratio,peak_hour_concentration,observed_hour_count,hourly_share_of_daily_volume,is_peak_hour,estimated_arrival_rate_vph,estimated_arrival_rate_vpm
0,10010625_WB,2015-05-14 00:00:00,286,Danforth Ave: Donlands Ave - Byron Ave,10010625,WB,2015-05-14,0,2015,5,3,0,17031,1809,0.106218,2.549234,24,0.016793,0,286,4.766667
1,10010625_WB,2015-05-14 01:00:00,198,Danforth Ave: Donlands Ave - Byron Ave,10010625,WB,2015-05-14,1,2015,5,3,0,17031,1809,0.106218,2.549234,24,0.011626,0,198,3.300000
2,10010625_WB,2015-05-14 02:00:00,134,Danforth Ave: Donlands Ave - Byron Ave,10010625,WB,2015-05-14,2,2015,5,3,0,17031,1809,0.106218,2.549234,24,0.007868,0,134,2.233333
3,10010625_WB,2015-05-14 03:00:00,118,Danforth Ave: Donlands Ave - Byron Ave,10010625,WB,2015-05-14,3,2015,5,3,0,17031,1809,0.106218,2.549234,24,0.006929,0,118,1.966667
4,10010625_WB,2015-05-14 04:00:00,131,Danforth Ave: Donlands Ave - Byron Ave,10010625,WB,2015-05-14,4,2015,5,3,0,17031,1809,0.106218,2.549234,24,0.007692,0,131,2.183333


### Note on Dataset Naming

The file `traffic_signal_hourly_dataset_2015_2019.csv` contains hourly traffic count data derived from the Toronto SVC dataset.

Despite the name, this dataset does **not yet include traffic signal information**. Signal mapping is performed later in `04_feature_engineering.ipynb`, where traffic count locations are spatially matched to the nearest signalized intersections.

The name is retained for consistency with earlier project stages and to avoid breaking downstream notebooks.


## 4) Persist Processed Traffic Flow Datasets

The daily dataset supports:
1. traffic demand analysis
2. recurring peak-hour analysis
3. location-level signal optimization screening

The hourly dataset is the direct bridge into downstream signal timing optimization and simulation notebooks.


In [4]:
daily.to_csv(daily_output_path, index=False)
hourly.to_csv(hourly_output_path, index=False)

print(f"Saved daily dataset: {daily_output_path}")
print(f"Saved hourly dataset: {hourly_output_path}")
print(f"Daily rows: {len(daily):,}")
print(f"Hourly rows: {len(hourly):,}")
print("Daily columns:", daily.columns.tolist())
print("Hourly columns:", hourly.columns.tolist())


Saved daily dataset: /Users/manavparikh/Documents/GitHub/unfc-capstone-traffic-forecasting/data/processed/modeling_dataset_2015_2019.csv
Saved hourly dataset: /Users/manavparikh/Documents/GitHub/unfc-capstone-traffic-forecasting/data/processed/traffic_signal_hourly_dataset_2015_2019.csv
Daily rows: 5,778
Hourly rows: 138,672
Daily columns: ['location_id', 'date', 'location_name', 'centreline_id', 'direction', 'year', 'month', 'day_of_week', 'is_weekend', 'daily_total_volume', 'peak_hour_volume', 'observed_hour_count', 'avg_observed_hourly_volume', 'peak_ratio', 'peak_hour_concentration', 'queue_pressure_proxy']
Hourly columns: ['location_id', 'hour', 'hourly_volume', 'location_name', 'centreline_id', 'direction', 'date', 'hour_of_day', 'year', 'month', 'day_of_week', 'is_weekend', 'daily_total_volume', 'peak_hour_volume', 'peak_ratio', 'peak_hour_concentration', 'observed_hour_count', 'hourly_share_of_daily_volume', 'is_peak_hour', 'estimated_arrival_rate_vph', 'estimated_arrival_rate_

## 5) Optional Validation Against Summary Data (Cross-Check Only)

This cross-check confirms that raw-derived aggregation is directionally consistent with the summary dataset. It does **not** define the optimization metrics.


In [5]:
if summary_path is None:
    print("Summary file not found; skipping optional cross-check.")
else:
    summary_df = pd.read_csv(summary_path, usecols=["count_id", "avg_daily_vol"])

    count_daily = (
        raw_df.assign(date=raw_df["time_start"].dt.floor("D"))
        .groupby(["count_id", "date"], as_index=False)
        .agg(raw_daily_total=("volume_15min", "sum"))
    )

    raw_count_avg = (
        count_daily.groupby("count_id", as_index=False)
        .agg(raw_avg_daily_vol=("raw_daily_total", "mean"))
    )

    merged_check = raw_count_avg.merge(
        summary_df.drop_duplicates(subset=["count_id"]),
        on="count_id",
        how="inner",
    )
    merged_check["abs_diff"] = (
        merged_check["raw_avg_daily_vol"] - merged_check["avg_daily_vol"]
    ).abs()
    merged_check["pct_diff"] = np.where(
        merged_check["avg_daily_vol"].abs() > 0,
        merged_check["abs_diff"] / merged_check["avg_daily_vol"].abs(),
        np.nan,
    )

    print(f"Cross-check matched count_ids: {len(merged_check):,}")
    if len(merged_check) > 0:
        print("Mean absolute diff:", round(merged_check["abs_diff"].mean(), 3))
        print("Median absolute diff:", round(merged_check["abs_diff"].median(), 3))
        print("Median pct diff:", round(merged_check["pct_diff"].median(), 4))
        display_cols = [
            "count_id",
            "raw_avg_daily_vol",
            "avg_daily_vol",
            "abs_diff",
            "pct_diff",
        ]
        display(merged_check.sort_values("abs_diff", ascending=False).head(10)[display_cols])


Cross-check matched count_ids: 1,026
Mean absolute diff: 0.205
Median absolute diff: 0.029
Median pct diff: 0.0


,count_id,raw_avg_daily_vol,avg_daily_vol,abs_diff,pct_diff
179,1681896,9809.000000,9947.7,138.700000,0.013943
828,2149871,2080.500000,2112.9,32.400000,0.015334
373,1692220,28944.333333,28961.0,16.666667,0.000575
909,2772588,6086.750000,6086.8,0.050000,0.000008
908,2772580,6787.250000,6787.3,0.050000,0.000007
903,2772540,7206.750000,7206.8,0.050000,0.000007
906,2772564,5450.250000,5450.3,0.050000,0.000009
904,2772548,6737.750000,6737.8,0.050000,0.000007
905,2772556,7743.750000,7743.8,0.050000,0.000006
727,2080684,7368.750000,7368.8,0.050000,0.000007


## Outputs

- `data/processed/modeling_dataset_2015_2019.csv`
  Daily traffic flow analysis dataset used for demand analysis and location-level peak-period summaries.
- `data/processed/traffic_signal_hourly_dataset_2015_2019.csv`
  Hourly intersection-level demand dataset for signal timing optimization and simulation setup.
